[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/3_Potato/code/baseline/solution.ipynb)

# Potato — baseline on Google Colab

Run all (GPU runtime recommended). The first cell fetches the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-potato) and recreates the contest file layout; every cell after it is the original, untouched baseline.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-potato", repo_type="dataset"))
ds = Path("dataset"); ds.mkdir(exist_ok=True)
for f in ("vocabulary.json","public_embeddings.npy","test_public.json"):
    if not (ds/f).exists(): os.symlink(DATA/"public"/f, ds/f)
print("ready: dataset/ ->", sorted(p.name for p in ds.iterdir()))


# Potato Contact — public baseline

This notebook contains a complete but simple baseline. It loads the public embeddings and proposes unused words that are close to the current winning word.

### Where should you make changes?

Your main work area is **Section 2: `PublicEmbeddingPlayer` — your solution**. That class decides which word to propose on every turn. You may rewrite its strategy, add methods, or change its parameters.

Sections 1 and 3 provide data loading and the contest input/output protocol. Run them, but normally do not change them. Section 4 explains how to export your solution.

## 1. Setup — run this cell, normally do not change it

This section loads `vocabulary.json` and `public_embeddings.npy` and prepares cosine similarities for the baseline.

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_DIR = Path(os.environ.get("POTATO_DATA_DIR", BASE_DIR / "dataset"))
WORDS_PATH = DATA_DIR / "vocabulary.json"
EMBEDDINGS_PATH = DATA_DIR / "public_embeddings.npy"

with WORDS_PATH.open() as file:
    words = json.load(file)

embeddings = np.load(EMBEDDINGS_PATH).astype(np.float32, copy=False)
if embeddings.ndim != 2 or embeddings.shape[0] != len(words):
    raise ValueError("Public embeddings are not aligned with the vocabulary")

norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1.0
normalized = embeddings / norms
similarities = normalized @ normalized.T
word_to_index = {word.casefold(): index for index, word in enumerate(words)}

print(
    f"Loaded {len(words)} words and embeddings with shape {embeddings.shape}",
    file=sys.stderr,
)

## 2. `PublicEmbeddingPlayer` — your solution

**This is the main cell you are expected to improve.** Everything in this cell is part of the baseline strategy.

You may replace the whole strategy, but keep these two parts of the interface:

1. the class must still be named `PublicEmbeddingPlayer`;
2. `respond(message)` must return one word from the provided vocabulary.

In [ ]:
# ===================== YOUR SOLUTION STARTS HERE =====================
# Change the parameters, methods, or the whole strategy in this cell.
import json
import sys
import numpy as np

class PublicEmbeddingPlayer:
    def __init__(self):
        self.proposed = set()

    def respond(self, message):
        champion = word_to_index[message["winner_word"].casefold()]
        # Pick the next unused word most similar to the winner
        for index in np.argsort(-similarities[champion]):
            index = int(index)
            if index not in self.proposed:
                self.proposed.add(index)
                return words[index]
        return words[0]

# ====================== YOUR SOLUTION ENDS HERE ======================

## 3. Contest protocol — run this cell, do not change it

This section connects your `PublicEmbeddingPlayer` to the judging system. Your program runs **once** and plays every game: it reads one JSON message per line, makes a fresh `PublicEmbeddingPlayer` at each `{"event": "new_game"}`, calls `player.respond(message)` on each turn, and exits on `{"event": "done"}`.

You normally should not edit this cell. Diagnostic output goes to `stderr`, so it does not interfere with the protocol. The final condition starts the loop only after the notebook has been exported to a `.py` file; running this cell inside Jupyter will not wait for terminal input.

In [ ]:
def run_interactive():
    # Your program runs ONCE and plays every game. Preparation (Section 1) already
    # ran above. A fresh PublicEmbeddingPlayer is created for each new game so its
    # per-game state resets, while the loaded embeddings/similarities are reused.
    player = PublicEmbeddingPlayer()
    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        try:
            message = json.loads(line)
        except json.JSONDecodeError:
            continue

        event = message.get("event")
        if event == "done":            # all games finished -> exit
            break
        if event == "new_game":        # a new game starts -> reset per-game state
            player = PublicEmbeddingPlayer()
            continue
        if "status" in message:        # this game ended (win/loss) -> wait for the next
            continue

        new_word = player.respond(message)
        print(
            f"turn={message['turn']} winner={message['winner_word']} proposal={new_word}",
            file=sys.stderr,
        )
        print(json.dumps({"new_word": new_word}), flush=True)


if "__file__" in globals():
    run_interactive()


## 4. Test and submit your solution

When you are happy with your changes in `PublicEmbeddingPlayer`:

1. Save `solution.ipynb`; this saved notebook is your official Contest submission;
2. Add the saved `solution.ipynb` to git, commit and push.  
3. Submit it through the Contest interface; it should be visible as the last commit.
4. Done — it will be tested on the private test set; you can see the result once it finishes.




Optionally, you can check the solution locally
```
python local_test.py solution.ipynb
```
The local tester uses public embeddings, so its score is approximate and may differ from the official private score. Do not add ordinary `print(...)` calls to standard output: the judge expects protocol JSON there. Send debugging messages to standard error with `print(..., file=sys.stderr)`.